# Verification of mod 49 classification of eigenforms
By default, recursive Nim verification reuses the source archives in `source_data/p7_mod49_transfer_maps/`: the Hecke matrices and, when available, the transfer maps and complement images. For older archives, it reconstructs only the missing presentation and transfer maps. It checks cyclic orders, equivariance and spanning before using inherited results. The source archives remain trusted producer outputs, not an independent replay of the Hecke construction.

Set `USE_RECURSIVE_NIM_VERIFICATION = False` for the Python full-source checks on the same archives. Build the native verifier once with `./build_verify_hecke_relations.sh`. The strong-realization check is unchanged.

Both verification sections use four persistent workers by default. Degrees run in ascending order within independent residue chains modulo 14 (both Dickson shifts are divisible by 14); completed results can appear out of order. Each chain includes lower untwisted plus cases, then the induction base in all six orientations. Each worker retains its recursive lower checks. Interrupting the loop stops the workers and their native children. The identities and strong-realization check are unchanged.


Each worker caches converted source records (64 MiB) and decoded native matrices (128 MiB matrix-entry budget), in addition to verified lower results. File changes invalidate Python preparation entries; native cache keys include the actual archived contents. No source archive is modified. These caches are bounded and live only for the verification session.


In [1]:
import json
import sys
from pathlib import Path

PYTHON_DIRECTORY = Path("python").resolve()
if str(PYTHON_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIRECTORY))

from hecke_congruences import *
from load_source_data import load_source_data
from p7_mod49 import p7_mod49_nim_relation_spec
from contextlib import closing
from verify_hecke_relations import verify_hecke_relations_nim, parallel_relation_cases

USE_RECURSIVE_NIM_VERIFICATION = True
VERIFICATION_WORKERS = 4

SOURCE_DATA_DIRECTORY = Path("source_data").resolve()
P7_MOD49_Q_SELECTOR_SOURCE_ARCHIVE = (
    SOURCE_DATA_DIRECTORY
    / "p7_mod49_transfer_maps" / "Q_selectors_mod343"
)
P7_MOD49_G_SOURCE_ARCHIVE = (
    SOURCE_DATA_DIRECTORY
    / "p7_mod49_transfer_maps" / "G_mod49"
)

USE_ARCHIVED_STRONG_SIGNATURES = True
STRONG_SIGNATURE_DIRECTORY = Path("strong_signatures/p7_m2").resolve()
P7_MOD49_STRONG_SIGNATURE_STATUS = (
    STRONG_SIGNATURE_DIRECTORY / "status.json"
)
P7_MOD49_STRONG_SIGNATURE_SUMMARY = (
    STRONG_SIGNATURE_DIRECTORY / "summary.json"
)

def show_verification_progress(event):
    """Display the current finite-source verification step and, when present, the linear-system dimension."""
    source = event.get("source", event)
    stage = event["stage"]
    if stage in ("passed", "failed"):
        return
    dimension = event.get("howell_dimension")
    detail = "" if dimension is None else f", Howell dimension={dimension}"
    print(
        f"d={source.get('degree')}, q={source.get('orientation')}: "
        f"{event.get('relation', '')} {stage}{detail}",
        end="\n" if stage != "start" else "\r",
        flush=True,
    )


# Identities for $Q_j(T_3)$ and the classification selectors

In [2]:
p = 7
m = 3
modulus = p^m
R = Integers(modulus)

S.<X> = PolynomialRing(R)

period = euler_phi(p^m)
a_m = p^m * (p - 1)
b_m = p^(m - 1) * (p + 1)
surjectivity_bound = a_m + b_m

exact_induction_base = tuple(
    range(b_m, surjectivity_bound, 2)
)

lower_base_degrees = tuple(
    range(0, b_m, 2)
)

F3 = {
    0: X^6 + 7*X^5 + 45*X^4 + 7*X^3 + 4*X^2 + 7*X,
    2: X^6 + 42*X^5 + 6*X^4 + 9*X^2 + 14*X,
    4: X^6 + 7*X^5 + 47*X^4 + 14*X^3 + X^2 + 28*X,
}

H = (X^7 - X)^3

classification_relations = p7_mod49_relation_polynomials()
R_rc = classification_relations["R"]
phi_rc = classification_relations["phi"]
psi_rc = classification_relations["psi"]

assert classification_relations["relation_count"] == 210
assert classification_relations["coordinate_relation_count"] == 126
assert classification_relations["joint_relation_count"] == 84

print("Dickson degrees:", a_m, b_m)
print("lower verification range:", lower_base_degrees)
print("exact induction base:", exact_induction_base)
print("coordinate polynomial ring:", classification_relations["coordinate_ring"])
print("example R_{40,1}^{(1)}:", R_rc[(40, 1, 1)])
print("phi_{40,1}:", phi_rc[(40, 1)])
print("psi_{40,1}:", psi_rc[(40, 1)])
print("classification relations:", classification_relations["relation_count"])

Dickson degrees: 2058 392
lower verification range: (0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98, 100, 102, 104, 106, 108, 110, 112, 114, 116, 118, 120, 122, 124, 126, 128, 130, 132, 134, 136, 138, 140, 142, 144, 146, 148, 150, 152, 154, 156, 158, 160, 162, 164, 166, 168, 170, 172, 174, 176, 178, 180, 182, 184, 186, 188, 190, 192, 194, 196, 198, 200, 202, 204, 206, 208, 210, 212, 214, 216, 218, 220, 222, 224, 226, 228, 230, 232, 234, 236, 238, 240, 242, 244, 246, 248, 250, 252, 254, 256, 258, 260, 262, 264, 266, 268, 270, 272, 274, 276, 278, 280, 282, 284, 286, 288, 290, 292, 294, 296, 298, 300, 302, 304, 306, 308, 310, 312, 314, 316, 318, 320, 322, 324, 326, 328, 330, 332, 334, 336, 338, 340, 342, 344, 346, 348, 350, 352, 354, 356, 358, 360, 362, 364, 366, 368, 370, 372, 374, 376, 378, 380, 382, 384, 386, 388, 390)
exact induction base: 

In [3]:
selector_source_path = P7_MOD49_Q_SELECTOR_SOURCE_ARCHIVE

if not USE_RECURSIVE_NIM_VERIFICATION:
    # Check that every required per-degree archive is present before verification.
    missing_degrees = [
        d for d in range(0, 2450, 2)
        if not (selector_source_path / f"degree_{d}.npz").is_file()
    ]
    if missing_degrees:
        raise FileNotFoundError(f"Missing source degrees: {missing_degrees}")
    
    selector_archive_probe = load_source_data(
        R, 0, 0, selector_source_path / "degree_0.npz",
    )
    assert selector_archive_probe["source_scope"] == "manin"
    assert set(selector_archive_probe["archived_hecke_matrices"]) == {3, 29}

def verify_Q_selector_case(d, q, session=None):
    """Check Q_j and the displayed selector relations on the oriented source at working precision 343.

    Use the configured native recursive route or the Sage route, retaining the
    literal order of the presented relation.
    """
    if USE_RECURSIVE_NIM_VERIFICATION:
        report = verify_hecke_relations_nim(
            p7_mod49_nim_relation_spec(d, q, 3),
            compute={
                "prime": 7, "exponent": 3, "degree": d, "orientation": q,
                "recursive": True, "recursive_verification": True,
                "archive_directory": str(selector_source_path),
            },
            session=session,
        )
        Q_test = dict(report["relations"][0], rank=report["rank"])
        selector_test = {
            "passed": report["passed"] is True,
            "relations": report["relations"][1:],
        }
        return {
            "degree": d, "orientation": q, "sign": (-1)^q,
            "relation_residue": (d + 2*q) % 6,
            "degree_residue": (d + 14*q) % 42,
            "Q": Q_test, "selectors": selector_test,
            "native": report, "passed": report["passed"] is True,
        }

    data = load_source_data(
        R,
        d,
        q,
        selector_source_path / f"degree_{d}.npz",
    )
    relation_residue = (d + 2*q) % 6

    Q_test = verify_divided_identities(
        F=H,
        Q=F3[relation_residue],
        n=3,
        a=2,
        b=1,
        data=data,
        check_descent=False,
    )

    selector_test = verify_p7_mod49_selector_identities(
        data=data,
        relation_data=classification_relations,
        check_descent=False,
    )

    return {
        "degree": d,
        "orientation": q,
        "sign": data["sign"],
        "relation_residue": relation_residue,
        "degree_residue": selector_test["degree_residue_mod42"],
        "Q": Q_test,
        "selectors": selector_test,
        "passed": (
            Q_test["passed"]
            and selector_test["passed"]
        ),
    }

In [4]:
cases = (
    [(d, 0) for d in sorted(lower_base_degrees)]
    + [
        (d, q)
        for d in sorted(exact_induction_base)
        for q in range(0, p - 1)
    ]
)
assert all(cases[i][0] <= cases[i+1][0] for i in range(len(cases)-1))

results = []
failures = []

# Keep each independent residue chain in one persistent worker.
assert a_m % 14 == 0 and b_m % 14 == 0
with closing(parallel_relation_cases(
    verify_Q_selector_case, cases, dependency_period=14,
    workers=VERIFICATION_WORKERS,
    native=USE_RECURSIVE_NIM_VERIFICATION,
    on_progress=show_verification_progress,
)) as completed:
    for test in completed:
        d, q = test["degree"], test["orientation"]
        results.append(test)

        Q_test = test["Q"]
        selector_test = test["selectors"]
        print(
            f"d={d:4d}, "
            f"r={test['degree_residue']:2d}, "
            f"q={q}, "
            f"sign={test['sign']:+d}, "
            f"rank={Q_test['rank']:4d}, "
            f"Q={Q_test['passed']}, "
            f"Q_route={Q_test['verification_route']}, "
            f"selectors={selector_test['passed']}, "
            f"recursive_route={test.get('native', {}).get('recursive_route', 'archive_full_source')}, "
            f"cached={test.get('native', {}).get('verification_cache_hit', False)}, "
            f"source_cache_hits={test.get('native', {}).get('source_preparation_cache', {}).get('hits', 0)}, "
            f"native_source_hits={test.get('native', {}).get('prepared_source_cache_hits', 0)}, "
            f"passed={test['passed']}",
            flush=True,
        )
        if not test["passed"]:
            failures.append(test)
            print(test)
            raise AssertionError("Q OR SELECTOR CHECK DID NOT PASS: inspect failed versus inconclusive")

assert len(results) == len(cases)
assert all(test["passed"] for test in results)
print("——————————————————————————————————————————————")
print("cases completed:", len(results))
print("failures:", len(failures))
print("ALL Q AND SELECTOR IDENTITIES VERIFIED")


d=0, q=0:  case_start
d=2, q=0:  case_start
d=4, q=0:  case_start
d=6, q=0:  case_start
d=   0, r= 0, q=0, sign=+1, rank=   0, Q=True, Q_route=explicit_witness_replay, selectors=True, recursive_route=direct_base, cached=False, source_cache_hits=0, native_source_hits=0, passed=True
d=8, q=0:  case_start
d=   4, r= 4, q=0, sign=+1, rank=   1, Q=True, Q_route=explicit_witness_replay, selectors=True, recursive_route=direct_base, cached=False, source_cache_hits=0, native_source_hits=0, passed=True
d=12, q=0:  case_start
d=   2, r= 2, q=0, sign=+1, rank=   1, Q=True, Q_route=explicit_witness_replay, selectors=True, recursive_route=direct_base, cached=False, source_cache_hits=0, native_source_hits=0, passed=True
d=10, q=0:  case_start
d=   6, r= 6, q=0, sign=+1, rank=   1, Q=True, Q_route=explicit_witness_replay, selectors=True, recursive_route=direct_base, cached=False, source_cache_hits=0, native_source_hits=0, passed=True
d=20, q=0:  case_start
d=8, q=0: k10_c5_L_roots shared_chain_howell,

KeyboardInterrupt: 

# Identities for $G_j(T_3,T_{29})$

In [ ]:
p = 7
m = 2
modulus = p^m
R = Integers(modulus)

S.<Z> = PolynomialRing(R)
J.<X,Y> = PolynomialRing(R, 2)

period = euler_phi(p^m)
a_m = p^m * (p - 1)
b_m = p^(m - 1) * (p + 1)
surjectivity_bound = a_m + b_m

exact_induction_base = tuple(
    range(b_m, surjectivity_bound, 2)
)

lower_base_degrees = tuple(
    range(0, b_m, 2)
)

alpha_beta = {
    0: (3, 4),
    2: (2, 5),
    4: (1, 6),
}

G_numerator = {
    r: Y - 2 - X*(X - alpha)*(X - beta)
    for r, (alpha, beta) in alpha_beta.items()
}

H = (Z^7 - Z)^3

print("Dickson degrees:", a_m, b_m)
print("lower verification range:", lower_base_degrees)
print("exact induction base:", exact_induction_base)

Dickson degrees: 294 56
lower verification range: (0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54)
exact induction base: (56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98, 100, 102, 104, 106, 108, 110, 112, 114, 116, 118, 120, 122, 124, 126, 128, 130, 132, 134, 136, 138, 140, 142, 144, 146, 148, 150, 152, 154, 156, 158, 160, 162, 164, 166, 168, 170, 172, 174, 176, 178, 180, 182, 184, 186, 188, 190, 192, 194, 196, 198, 200, 202, 204, 206, 208, 210, 212, 214, 216, 218, 220, 222, 224, 226, 228, 230, 232, 234, 236, 238, 240, 242, 244, 246, 248, 250, 252, 254, 256, 258, 260, 262, 264, 266, 268, 270, 272, 274, 276, 278, 280, 282, 284, 286, 288, 290, 292, 294, 296, 298, 300, 302, 304, 306, 308, 310, 312, 314, 316, 318, 320, 322, 324, 326, 328, 330, 332, 334, 336, 338, 340, 342, 344, 346, 348)


In [ ]:
G_source_path = P7_MOD49_G_SOURCE_ARCHIVE

if not USE_RECURSIVE_NIM_VERIFICATION:
    # Check that every required per-degree archive is present before verification.
    missing_degrees = [
        d for d in range(0, 350, 2)
        if not (G_source_path / f"degree_{d}.npz").is_file()
    ]
    if missing_degrees:
        raise FileNotFoundError(f"Missing source degrees: {missing_degrees}")
    
    G_archive_probe = load_source_data(
        R, 0, 0, G_source_path / "degree_0.npz",
    )
    assert G_archive_probe["source_scope"] == "manin"
    assert set(G_archive_probe["archived_hecke_matrices"]) == {3, 29}

def verify_G_case(d, q, session=None):
    """Check the displayed G_j division relation and its terminal polynomial in this degree and orientation.

    Use the configured source at working precision 49 and return its finite verification report.
    """
    if USE_RECURSIVE_NIM_VERIFICATION:
        report = verify_hecke_relations_nim(
            p7_mod49_nim_relation_spec(d, q, 2),
            compute={
                "prime": 7, "exponent": 2, "degree": d, "orientation": q,
                "recursive": True, "recursive_verification": True,
                "archive_directory": str(G_source_path),
            },
            session=session,
        )
        G_test = dict(report["relations"][0], rank=report["rank"])
        return {
            "degree": d, "orientation": q, "sign": (-1)^q,
            "relation_residue": (d + 2*q) % 6,
            "G": G_test, "native": report,
            "passed": report["passed"] is True,
        }

    data = load_source_data(
        R,
        d,
        q,
        G_source_path / f"degree_{d}.npz",
    )

    relation_residue = (d + 2*q) % 6

    test = verify_divided_joint_identities(
        F=H,
        Q=G_numerator[relation_residue],
        hecke_indices=(3, 29),
        a=1,
        b=1,
        data=data,
        check_descent=False,
    )

    return {
        "degree": d,
        "orientation": q,
        "sign": data["sign"],
        "relation_residue": relation_residue,
        "G": test,
        "passed": test["passed"],
    }

In [ ]:
cases = (
    [(d, 0) for d in sorted(lower_base_degrees)]
    + [
        (d, q)
        for d in sorted(exact_induction_base)
        for q in range(0, p - 1)
    ]
)
assert all(cases[i][0] <= cases[i+1][0] for i in range(len(cases)-1))

results = []
failures = []

# Keep each independent residue chain in one persistent worker.
assert a_m % 14 == 0 and b_m % 14 == 0
with closing(parallel_relation_cases(
    verify_G_case, cases, dependency_period=14,
    workers=VERIFICATION_WORKERS,
    native=USE_RECURSIVE_NIM_VERIFICATION,
    on_progress=show_verification_progress,
)) as completed:
    for test in completed:
        d, q = test["degree"], test["orientation"]
        results.append(test)

        G_test = test["G"]
        print(
            f"d={d:3d}, "
            f"r={test['relation_residue']:2d}, "
            f"q={q}, "
            f"sign={test['sign']:+d}, "
            f"rank={G_test['rank']:3d}, "
            f"G={G_test['passed']}, "
            f"route={G_test['verification_route']}, "
            f"recursive_route={test.get('native', {}).get('recursive_route', 'archive_full_source')}, "
            f"cached={test.get('native', {}).get('verification_cache_hit', False)}, "
            f"source_cache_hits={test.get('native', {}).get('source_preparation_cache', {}).get('hits', 0)}, "
            f"native_source_hits={test.get('native', {}).get('prepared_source_cache_hits', 0)}, "
            f"passed={test['passed']}",
            flush=True,
        )
        if not test["passed"]:
            failures.append(test)
            print(test)
            raise AssertionError("G CHECK DID NOT PASS: inspect failed versus inconclusive")

assert len(results) == len(cases)
assert all(test["passed"] for test in results)
print("——————————————————————————————————————————————")
print("cases completed:", len(results))
print("failures:", len(failures))
print("ALL G IDENTITIES VERIFIED")


# Check that each possible signature given by the identities above is realized by a strong eigenform

In [ ]:
F7_field = GF(7)
weight_period = 42

tangent_coefficients = {
    0: (0, 7, 4, 7, 45, 7, 1),
    2: (0, 14, 9, 0, 6, 42, 1),
    4: (0, 28, 1, 14, 47, 7, 1),
}

centres = {
    0: (0, 3, 4),
    2: (0, 2, 5),
    4: (0, 1, 6),
}

def evaluate_classification_relation(record, W3, W29, x, y):
    """Evaluate the recorded coordinate or graph polynomial on the proposed scalar signature."""
    polynomial = record["polynomial"]

    if record["family"] == "compact_W_coordinate":
        return polynomial(W3, W29)

    if record["raw_digit"] == "U":
        return polynomial(W3, W29, x)

    return polynomial(W3, W29, y)

possible_signatures = {}
possible_signature_triples = {}

for k_residue in range(0, weight_period, 2):
    degree_residue = (k_residue - 2) % weight_period
    relation_residue = degree_residue % 6
    possible_signatures[k_residue] = set()
    possible_signature_triples[k_residue] = set()

    alpha, beta = alpha_beta[relation_residue]

    for c in centres[relation_residue]:
        branch_relations = classification_relations["by_branch"][
            (degree_residue, c)
        ]

        for x in range(7):
            for y in range(7):
                a3 = c + 7*x
                a29 = 2 + 7*y

                F3_value = sum(
                    coefficient*a3^power
                    for power, coefficient
                    in enumerate(tangent_coefficients[relation_residue])
                )
                assert F3_value % 49 == 0
                W3 = F7_field(F3_value // 49)

                G_value = (
                    a29 - 2
                    - a3*(a3 - alpha)*(a3 - beta)
                )
                assert G_value % 7 == 0
                W29 = F7_field(G_value // 7)

                if all(
                    evaluate_classification_relation(
                        record, W3, W29, F7_field(x), F7_field(y)
                    ) == 0
                    for record in branch_relations
                ):
                    possible_signature_triples[k_residue].add(
                        (c, x, y)
                    )
                    possible_signatures[k_residue].add(
                        (a3, a29)
                    )

    assert len(possible_signatures[k_residue]) == 15

assert sum(len(values) for values in possible_signatures.values()) == 315

print(
    f"{'k mod ' + str(weight_period):<10} | "
    f"(a_3, a_29) mod 49"
)
print("-" * 120)

for r in sorted(possible_signatures):
    pairs = ", ".join(
        f"({a3}, {a29})"
        for a3, a29 in sorted(possible_signatures[r])
    )

    print(f"{str(r):<10} | {pairs}")

k mod 42   | (a_3, a_29) mod 49
------------------------------------------------------------------------------------------------------------------------
0          | (0, 44), (1, 23), (7, 9), (8, 23), (13, 23), (14, 2), (15, 23), (21, 23), (28, 23), (34, 23), (35, 2), (36, 23), (41, 23), (42, 9), (48, 23)
2          | (0, 2), (3, 30), (4, 30), (7, 9), (10, 30), (14, 30), (21, 16), (24, 30), (25, 30), (28, 16), (35, 30), (39, 30), (42, 9), (45, 30), (46, 30)
4          | (0, 30), (2, 37), (5, 37), (7, 9), (12, 37), (14, 44), (21, 37), (23, 37), (26, 37), (28, 37), (35, 44), (37, 37), (42, 9), (44, 37), (47, 37)
6          | (0, 30), (1, 44), (6, 44), (7, 44), (8, 44), (14, 37), (20, 44), (21, 9), (28, 9), (29, 44), (35, 37), (41, 44), (42, 44), (43, 44), (48, 44)
8          | (0, 2), (3, 2), (4, 2), (7, 9), (11, 2), (14, 30), (17, 2), (21, 16), (28, 16), (32, 2), (35, 30), (38, 2), (42, 9), (45, 2), (46, 2)
10         | (0, 44), (5, 9), (7, 23), (9, 9), (14, 9), (16, 9), (21, 2), (23, 9

In [ ]:
def krw_reduction_to_integer(x, nf, pr, p, m):
    """Find an integer representative of the selected coefficient modulo the KRW ideal.

    Test equality in the prime-ideal quotient rather than assuming the residue is rational.
    """
    e = pr[2]
    N = e*(m - 1) + 1

    candidates = [
        r
        for r in range(p^m)
        if nf.idealval(x - r, pr) >= N
    ]

    if len(candidates) != 1:
        raise ValueError(
            f"expected one residue in Z/{p^m}Z, "
            f"but found {candidates}"
        )

    return Integers(p^m)(candidates[0])

In [ ]:
B = 380
p = 7
m = 2
l1, l2 = 3, 29

R = Integers(p^m)
signatures = []

if USE_ARCHIVED_STRONG_SIGNATURES:
    status = json.loads(
        P7_MOD49_STRONG_SIGNATURE_STATUS.read_text(encoding="utf-8")
    )
    summary = json.loads(
        P7_MOD49_STRONG_SIGNATURE_SUMMARY.read_text(encoding="utf-8")
    )

    assert status["schema"] == "hecke.strong-signature-status.v1"
    assert status["state"] == "completed"
    assert status["prime"] == p
    assert status["exponent"] == m
    assert status["maximum_weight"] == B
    assert status["bounded_scan_complete"]
    assert status["failed"] == []

    assert summary["schema"] == "hecke.strong-signature-summary.v1"
    assert summary["prime"] == p
    assert summary["exponent"] == m
    assert summary["hecke_indices"] == [l1, l2]
    assert summary["maximum_weight"] == B
    assert summary["bounded_scan_complete"]
    assert summary["completed_weights"] == list(range(2, B + 1, 2))
    assert summary["nonrational_packet_count"] == 0
    assert summary["rational_signature_count"] == len(
        summary["rational_signatures"]
    )

    for entry in summary["rational_signatures"]:
        k = ZZ(entry["weight"])
        residues = entry["eigenvalue_residues"]

        assert ZZ(entry["weight_residue"]) == k % weight_period
        assert len(residues) == 2

        signatures.append((
            k,
            R(ZZ(residues[0])),
            R(ZZ(residues[1])),
        ))

    print(
        f"loaded {len(signatures)} distinct strong signatures "
        f"through weight {B}"
    )

else:
    for k in range(12, B + 1, 2):
        S = CuspForms(1, k)

        if S.dimension() == 0:
            continue

        print(f"calculating strong signatures at weight {k}")

        for j, f in enumerate(S.newforms(names='a')):
            K = f.base_ring()

            # Rational eigenform orbit
            if K == QQ:
                signature = (
                    k,
                    R(f[l1]),
                    R(f[l2]),
                )
                signatures.append(signature)
                print(signature)
                continue

            # Nonrational eigenform orbit
            pol = K.pari_polynomial('y')
            nf = pari([pol, [p]]).nfinit(4)

            a1 = pari(f[l1])
            a2 = pari(f[l2])

            for pr in nf.idealprimedec(p):
                a1_bar = krw_reduction_to_integer(
                    a1, nf, pr, p, m
                )

                a2_bar = krw_reduction_to_integer(
                    a2, nf, pr, p, m
                )

                signature = (k, a1_bar, a2_bar)
                signatures.append(signature)

loaded 315 distinct strong signatures through weight 380


In [ ]:
signatures_by_weight_residue = {}

for k, a3, a29 in signatures:
    r = k % weight_period

    if r not in signatures_by_weight_residue:
        signatures_by_weight_residue[r] = set()

    signatures_by_weight_residue[r].add((ZZ(a3), ZZ(a29)))

In [ ]:
print(
    f"{'k mod ' + str(weight_period):<10} | "
    f"(a_{l1}, a_{l2}) mod {p^m}"
)
print("-" * 120)

for r in sorted(signatures_by_weight_residue):
    pairs = ", ".join(
        f"({a1}, {a2})"
        for a1, a2 in sorted(signatures_by_weight_residue[r])
    )

    print(f"{str(r):<10} | {pairs}")

k mod 42   | (a_3, a_29) mod 49
------------------------------------------------------------------------------------------------------------------------
0          | (0, 44), (1, 23), (7, 9), (8, 23), (13, 23), (14, 2), (15, 23), (21, 23), (28, 23), (34, 23), (35, 2), (36, 23), (41, 23), (42, 9), (48, 23)
2          | (0, 2), (3, 30), (4, 30), (7, 9), (10, 30), (14, 30), (21, 16), (24, 30), (25, 30), (28, 16), (35, 30), (39, 30), (42, 9), (45, 30), (46, 30)
4          | (0, 30), (2, 37), (5, 37), (7, 9), (12, 37), (14, 44), (21, 37), (23, 37), (26, 37), (28, 37), (35, 44), (37, 37), (42, 9), (44, 37), (47, 37)
6          | (0, 30), (1, 44), (6, 44), (7, 44), (8, 44), (14, 37), (20, 44), (21, 9), (28, 9), (29, 44), (35, 37), (41, 44), (42, 44), (43, 44), (48, 44)
8          | (0, 2), (3, 2), (4, 2), (7, 9), (11, 2), (14, 30), (17, 2), (21, 16), (28, 16), (32, 2), (35, 30), (38, 2), (42, 9), (45, 2), (46, 2)
10         | (0, 44), (5, 9), (7, 23), (9, 9), (14, 9), (16, 9), (21, 2), (23, 9

In [ ]:
assert possible_signatures == signatures_by_weight_residue

print("EVERY POSSIBLE MODULO-49 SIGNATURE IS REALIZED")

EVERY POSSIBLE MODULO-49 SIGNATURE IS REALIZED
